# 02 Cleaning the Data

In this notebook, I'll use the script I wrote earlier to clean the data. It's better to keep the cleaning code in a separate file so I can use it again if I need to. I'll also check if everything looks right after cleaning.

### Imports

Load the libraries used for cleaning:
- `pandas` for DataFrame transformations.
- `numpy` for numerical thresholds and filtering logic.

In [26]:
import pandas as pd
import numpy as np

### Load Raw Dataset

Read the extracted raw CSV into `df` so all cleaning operations are applied on one in-memory table.

In [27]:
df = pd.read_csv("../data/raw/APY.csv")

### Normalize Column Names

Standardize column headers to avoid schema inconsistencies:
- strip extra spaces,
- convert names to lowercase,
- rename `crop_year` to `year` for cleaner grouping later.

In [28]:
df.columns = df.columns.str.strip().str.lower()
df.rename(columns={
    "crop_year": "year"
}, inplace=True)

### Clean Key Text Fields

Trim leading/trailing spaces in `state`, `district`, `crop`, and `season` so category values do not split into duplicates due to whitespace.

In [29]:
for col in ["state", "district", "crop", "season"]:
    df[col] = df[col].str.strip()

### Remove Critical Missing Values

Drop rows where `area` or `production` is missing, because these fields are required for yield and trend calculations.

In [30]:
df = df.dropna(subset=["area", "production"])

### Apply Basic Validity Filters

Keep only logically valid records:
- `area` must be greater than zero,
- `production` must be non-negative.

In [31]:
df = df[(df["area"] > 0) & (df["production"] >= 0)]

### Standardize Season Labels

Convert season names to title case (for example, `kharif` to `Kharif`) so grouping and charts use consistent category names.

In [32]:
df["season"] = df["season"].str.title()

### Remove Duplicate Rows

Drop exact duplicates to prevent repeated records from biasing aggregates and summary metrics.

In [33]:
df = df.drop_duplicates()

### Keep Years with Sufficient Coverage

Count records per year, then keep only years with at least 80% of the maximum yearly record count.

This reduces sparsely sampled years that can distort trend analysis.

In [34]:
year_counts = df["year"].value_counts()
threshold = 0.8 * year_counts.max()

valid_years = year_counts[year_counts >= threshold].index
df = df[df["year"].isin(valid_years)]

### Exclude Coconut Records

Remove `Coconut` rows because this crop is typically recorded in counts (nuts), which is not directly comparable with tonne-based production values.

In [35]:
df = df[df["crop"] != "Coconut"]

### Recompute Yield

Calculate yield as:

`yield = production / area`

This ensures yield is internally consistent with the cleaned `production` and `area` values.

In [36]:
df["yield"] = df["production"] / df["area"]

### Save Cleaned Dataset

Write the final cleaned table to `../data/processed/cleaned_data.csv` so downstream EDA and statistical notebooks use a stable prepared dataset.

In [37]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)